# Toward a Universal Agent Instruction File

**Primary contribution:** a data-driven template for agent instruction files, grounded in three evidence layers from AIDev + local repository snapshots.

| Layer | RQ | Question | Template input |
|-------|----|----------|----------------|
| 1 Prevalence | RQ1 | Who adopts which formats? Multi-file drift? | Domain-specific sections; duplication = essential content |
| 2 Content | RQ2 | Shared core / unique extensions / gaps? | REQUIRED vs optional vs recommended additions |
| 3 Maintenance | RQ3 | What survives, what is abandoned, what triggers updates? | Durable = REQUIRED; ephemeral = optional; add maintenance guidance |

**Four file types (operationalized):**
1. `AGENTS.md` — vendor-agnostic
2. `CLAUDE.md` — Claude Code
3. Cursor rules — `.cursorrules` or `.cursor/rules/**`
4. Copilot instructions — `.github/copilot-instructions.md` or `*.instructions.md`

> Note: `GEMINI.md` basename matches are rare and often false positives (e.g. docs about Gemini); excluded from the core four unless path basename is exactly `GEMINI.md` at repo root or package root.

## 0. Setup & data inventory

In [ ]:
from __future__ import annotations

import ast
import csv
import os
import re
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

csv.field_size_limit(10**9)

BASE = Path("/gpuhome/shanto1/Agents Instruction/ai_agent_instruction_files")
TOP_CSV = BASE / "top_500_projects_with_files.csv"
BOTTOM_CSV = BASE / "bottom_500_projects_with_files.csv"
TOP_REPOS = BASE / "top_repos"
BOTTOM_REPOS = BASE / "bottom_repos"

FILE_TYPES = [
    "AGENTS.md",
    "CLAUDE.md",
    "Cursor rules",
    "Copilot instructions",
]

print("CSV present:", TOP_CSV.exists(), BOTTOM_CSV.exists())
print("Cloned top/bottom repos:",
      sum(1 for p in TOP_REPOS.iterdir() if p.is_dir()) if TOP_REPOS.exists() else 0,
      sum(1 for p in BOTTOM_REPOS.iterdir() if p.is_dir()) if BOTTOM_REPOS.exists() else 0)

In [ ]:
def classify_path(path: str) -> set[str]:
    """Map a repo-relative path to zero or more of the four instruction file types."""
    p = path.replace("\\", "/")
    pl = p.lower()
    name = Path(pl).name
    found: set[str] = set()

    if name == "agents.md":
        found.add("AGENTS.md")
    if name == "claude.md":
        found.add("CLAUDE.md")
    if name == ".cursorrules" or "/.cursor/rules/" in pl or pl.endswith("/.cursor/rules"):
        found.add("Cursor rules")
    if name == "copilot-instructions.md" or name.endswith(".instructions.md") or "/.github/instructions/" in pl:
        found.add("Copilot instructions")
    return found


def load_project_table(csv_path: Path, cohort: str) -> pd.DataFrame:
    rows = []
    with open(csv_path, newline="", encoding="utf-8", errors="replace") as f:
        for row in csv.DictReader(f):
            try:
                files = ast.literal_eval(row["files"]) if row.get("files") else []
            except Exception:
                files = []
            types = set()
            matched_paths = defaultdict(list)
            for fp in files:
                hits = classify_path(str(fp))
                for t in hits:
                    types.add(t)
                    matched_paths[t].append(fp)
            rows.append({
                "cohort": cohort,
                "id": int(row["id"]) if row.get("id") else None,
                "full_name": row.get("full_name"),
                "language": row.get("language") or "Unknown",
                "license": row.get("license") or "Unknown",
                "stars": float(row["stars"]) if row.get("stars") else np.nan,
                "forks": float(row["forks"]) if row.get("forks") else np.nan,
                "n_files_listed": len(files),
                "types": frozenset(types),
                "n_types": len(types),
                "has_any": len(types) > 0,
                "matched_paths": dict(matched_paths),
                **{f"has_{t}": (t in types) for t in FILE_TYPES},
            })
    return pd.DataFrame(rows)


projects = pd.concat([
    load_project_table(TOP_CSV, "top"),
    load_project_table(BOTTOM_CSV, "bottom"),
], ignore_index=True)

print(projects.groupby("cohort").agg(repos=("full_name", "count"), adopters=("has_any", "sum")))
projects.head(3)

## RQ1 — Prevalence & Adoption

**RQ1:** How prevalent are agent instruction files across repository domains, and what types of repositories use which file formats?

**Methods (this notebook):** frequency tables; language × format cross-tabs; multi-file coexistence; logistic regression on repo characteristics.

**Still needs AIDev tables for full paper claims:** `pr_commit_details` + `pull_request` for *when* files appear in agentic PRs, and sibling-file drift lag from commit timestamps.

In [ ]:
def prevalence_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for cohort, g in df.groupby("cohort"):
        n = len(g)
        row = {"cohort": cohort, "n_repos": n, "any_instruction_file": int(g["has_any"].sum()),
               "any_pct": 100 * g["has_any"].mean()}
        for t in FILE_TYPES:
            col = f"has_{t}"
            row[f"{t}_n"] = int(g[col].sum())
            row[f"{t}_pct"] = 100 * g[col].mean()
        rows.append(row)
    return pd.DataFrame(rows)


rq1_prev = prevalence_table(projects)
rq1_prev

In [ ]:
# Multi-file coexistence (redundancy the universal template aims to solve)
coexist = (
    projects.groupby(["cohort", "n_types"])
    .size()
    .unstack(fill_value=0)
)
print("Repos with N distinct instruction formats:")
display(coexist)

pair_counts = Counter()
for types in projects.loc[projects["n_types"] >= 2, "types"]:
    for a, b in combinations(sorted(types), 2):
        pair_counts[(a, b)] += 1

pair_df = (
    pd.DataFrame([{"type_a": a, "type_b": b, "n_repos": n} for (a, b), n in pair_counts.items()])
    .sort_values("n_repos", ascending=False)
    .reset_index(drop=True)
)
print("\nMost common co-occurring format pairs:")
pair_df

In [ ]:
# Domain (primary language) × format — among adopters
adopters = projects[projects["has_any"]].copy()

lang_fmt = []
for cohort, g in adopters.groupby("cohort"):
    top_langs = g["language"].value_counts().head(12).index
    sub = g[g["language"].isin(top_langs)]
    for lang, lg in sub.groupby("language"):
        row = {"cohort": cohort, "language": lang, "n": len(lg)}
        for t in FILE_TYPES:
            row[t] = 100 * lg[f"has_{t}"].mean()
        lang_fmt.append(row)

lang_fmt_df = pd.DataFrame(lang_fmt).sort_values(["cohort", "n"], ascending=[True, False])
lang_fmt_df.round(1)

In [ ]:
# Logistic regression: P(has format) ~ log1p(stars) + log1p(forks) + cohort + language (top langs)
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf

    reg_df = projects.copy()
    top_langs = reg_df["language"].value_counts().head(8).index
    reg_df["lang_group"] = reg_df["language"].where(reg_df["language"].isin(top_langs), "Other")
    reg_df["log_stars"] = np.log1p(reg_df["stars"].fillna(0))
    reg_df["log_forks"] = np.log1p(reg_df["forks"].fillna(0))
    reg_df["is_top"] = (reg_df["cohort"] == "top").astype(int)

    logit_summaries = {}
    for t in FILE_TYPES:
        ycol = f"has_{t}"
        # formula-safe column name
        reg_df["y"] = reg_df[ycol].astype(int)
        model = smf.logit("y ~ log_stars + log_forks + is_top + C(lang_group)", data=reg_df).fit(disp=False)
        logit_summaries[t] = model
        print("=" * 72)
        print(f"Outcome: has {t}")
        print(model.summary2().tables[1].round(3))
except ImportError:
    print("statsmodels not installed — skipping logistic regression.")
    print("Install with: pip install statsmodels")

### RQ1b — Drift lag across sibling files (needs git history)

For repos with ≥2 formats, measure days between last commits touching each sibling file.
High lag ⇒ multi-file maintenance burden (motivation for a universal template).

In [ ]:
import subprocess
from datetime import datetime, timezone


def repo_dir_for(full_name: str, cohort: str) -> Path | None:
    slug = full_name.replace("/", "_")
    root = TOP_REPOS if cohort == "top" else BOTTOM_REPOS
    cand = root / slug
    return cand if cand.is_dir() else None


def last_commit_ts(repo: Path, rel_path: str) -> datetime | None:
    try:
        out = subprocess.check_output(
            ["git", "-C", str(repo), "log", "-1", "--format=%cI", "--", rel_path],
            stderr=subprocess.DEVNULL,
            text=True,
            timeout=30,
        ).strip()
    except Exception:
        return None
    if not out:
        return None
    return datetime.fromisoformat(out.replace("Z", "+00:00"))


def sibling_drift_for_row(row, max_paths_per_type: int = 3) -> list[dict]:
    """Compute pairwise |Δdays| between latest commits of sibling instruction formats."""
    repo = repo_dir_for(row["full_name"], row["cohort"])
    if repo is None or row["n_types"] < 2:
        return []
    type_ts = {}
    for t, paths in row["matched_paths"].items():
        times = []
        for p in paths[:max_paths_per_type]:
            ts = last_commit_ts(repo, p)
            if ts is not None:
                times.append(ts)
        if times:
            type_ts[t] = max(times)
    records = []
    for a, b in combinations(sorted(type_ts), 2):
        lag_days = abs((type_ts[a] - type_ts[b]).total_seconds()) / 86400
        records.append({
            "full_name": row["full_name"],
            "cohort": row["cohort"],
            "type_a": a,
            "type_b": b,
            "lag_days": lag_days,
            "ts_a": type_ts[a].isoformat(),
            "ts_b": type_ts[b].isoformat(),
        })
    return records


# Demo on a small sample first (full run can take minutes)
multi = projects[projects["n_types"] >= 2].copy()
sample = multi.head(25)
drift_rows = []
for _, row in sample.iterrows():
    drift_rows.extend(sibling_drift_for_row(row))

drift_sample = pd.DataFrame(drift_rows)
print(f"Sampled {len(sample)} multi-format repos → {len(drift_sample)} pairwise lag observations")
if len(drift_sample):
    display(drift_sample.groupby(["type_a", "type_b"])["lag_days"].describe().round(1))
    display(drift_sample.sort_values("lag_days", ascending=False).head(10))
else:
    print("No git history found for sample (repos may be shallow / incomplete clones).")

## RQ2 — Content Similarity (scaffold)

**RQ2:** How similar are agent instruction files in content — shared core, format-unique extensions, and consistent gaps?

| Step | Method | Template mapping |
|------|--------|------------------|
| Extract headings | `#` / `##` markdown headers (+ Cursor frontmatter `description`) | Section vocabulary |
| Jaccard | Pairwise similarity across formats | Shared vs divergent structure |
| Card sort | 16 instruction categories, 2 coders, Cohen's κ | Taxonomy |
| Core / unique / gap | Present in ≥3 types / exactly 1 / 0 types | REQUIRED / optional / recommended |

In [ ]:
HEADING_RE = re.compile(r"^(#{1,3})\s+(.+?)\s*$", re.M)

SKIP_DIR_NAMES = {
    ".git", "node_modules", "dist", "build", ".venv", "venv",
    "__pycache__", ".tox", "target", "vendor",
}


def iter_instruction_files(repo_root: Path):
    for dirpath, dirnames, filenames in os.walk(repo_root):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIR_NAMES]
        for fn in filenames:
            full = Path(dirpath) / fn
            rel = str(full.relative_to(repo_root)).replace("\\", "/")
            types = classify_path(rel)
            if types:
                yield rel, types, full


def extract_headings(text: str) -> list[str]:
    heads = []
    for m in HEADING_RE.finditer(text):
        h = re.sub(r"\s+", " ", m.group(2)).strip().lower()
        h = re.sub(r"[^a-z0-9 /+_.-]+", "", h)
        if h:
            heads.append(h)
    return heads


def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def collect_heading_sets(repo_roots: list[Path], limit_repos: int | None = 80) -> dict[str, set[str]]:
    """Union of normalized headings observed per file type."""
    by_type: dict[str, set[str]] = {t: set() for t in FILE_TYPES}
    n = 0
    for root in repo_roots:
        if not root.is_dir():
            continue
        for repo in sorted(root.iterdir()):
            if not repo.is_dir():
                continue
            n += 1
            if limit_repos is not None and n > limit_repos:
                return by_type
            for rel, types, full in iter_instruction_files(repo):
                try:
                    text = full.read_text(encoding="utf-8", errors="replace")
                except Exception:
                    continue
                heads = extract_headings(text)
                for t in types:
                    by_type[t].update(heads)
    return by_type


# Pilot on first N top repos (raise limit_repos=None for full pass)
heading_sets = collect_heading_sets([TOP_REPOS], limit_repos=60)
print({t: len(s) for t, s in heading_sets.items()})

jacc_rows = []
for a, b in combinations(FILE_TYPES, 2):
    jacc_rows.append({"type_a": a, "type_b": b, "jaccard_headings": jaccard(heading_sets[a], heading_sets[b]),
                      "|A|": len(heading_sets[a]), "|B|": len(heading_sets[b]),
                      "|A∩B|": len(heading_sets[a] & heading_sets[b])})
pd.DataFrame(jacc_rows).sort_values("jaccard_headings", ascending=False)

In [ ]:
# Shared core (≥3 types), unique (exactly 1), and crude lexical gaps
from collections import defaultdict as dd

heading_type_count = dd(set)
for t, heads in heading_sets.items():
    for h in heads:
        heading_type_count[h].add(t)

shared_core = sorted([h for h, ts in heading_type_count.items() if len(ts) >= 3])
unique = {t: sorted([h for h, ts in heading_type_count.items() if ts == {t}])[:25]
          for t in FILE_TYPES}

print(f"Shared-core headings (≥3 types): {len(shared_core)}")
print(shared_core[:40])

# Keyword probes for critical gaps often missing in agent docs
GAP_PROBES = {
    "security": ["security", "secret", "credential", "vulnerability", "owasp"],
    "performance": ["performance", "latency", "profiling", "benchmark"],
    "accessibility": ["accessibility", "a11y", "wcag"],
    "licensing": ["license", "copyright", "spdx"],
    "maintenance": ["stale", "outdated", "keep updated", "maintenance"],
}

all_heads = " ".join(heading_type_count.keys())
gap_hits = {
    theme: [kw for kw in kws if kw in all_heads]
    for theme, kws in GAP_PROBES.items()
}
print("\nGap probe hits in headings (empty ⇒ candidate recommended addition):")
for theme, hits in gap_hits.items():
    print(f"  {theme}: {hits or 'MISSING'}")

### RQ2 qualitative protocol (for paper)

**Open card sorting — draft 16 categories** (refine after pilot):

1. Project overview / purpose  
2. Repository structure  
3. Build / run / install  
4. Test / CI commands  
5. Code style / formatting  
6. Architecture / design constraints  
7. Domain models / key abstractions  
8. API / interface contracts  
9. Do / Don't rules  
10. Security / secrets  
11. Performance  
12. Git / PR / commit conventions  
13. Tooling-specific agent config  
14. Examples / few-shot snippets  
15. Ownership / contacts  
16. Maintenance / when to update this file  

**Reliability:** two coders independently label a stratified sample (by format × cohort); report Cohen's κ; resolve disagreements by discussion.

## RQ3 — Maintenance Dynamics (scaffold)

**RQ3:** How do agent instruction files evolve — durable vs ephemeral content, and what triggers updates?

| Metric | Source | Notes |
|--------|--------|-------|
| Churn / update frequency | git log on cloned repos **or** AIDev `pr_commit_details` filtered to instruction paths | Per format + per content category |
| Ownership | `git shortlog` / commit authors | Concentration vs shared |
| Staleness lag | time since last touch vs repo activity | |
| Temporal pattern | burst-taper / continuous / abandoned | Fit on commit-time series |
| Triggers | reactive (post-failure CI/review) vs proactive co-change | Needs `pr_reviews`, `pr_task_type` from AIDev |

**Stats planned:** Kruskal-Wallis across formats; Cliff's δ effect sizes; dual-coder trigger labels.

In [ ]:
def commit_count_for_path(repo: Path, rel_path: str) -> int:
    try:
        out = subprocess.check_output(
            ["git", "-C", str(repo), "rev-list", "--count", "HEAD", "--", rel_path],
            stderr=subprocess.DEVNULL,
            text=True,
            timeout=30,
        ).strip()
        return int(out) if out else 0
    except Exception:
        return 0


def maintenance_pilot(cohort: str = "top", n_repos: int = 40) -> pd.DataFrame:
    root = TOP_REPOS if cohort == "top" else BOTTOM_REPOS
    rows = []
    for i, repo in enumerate(sorted(p for p in root.iterdir() if p.is_dir())):
        if i >= n_repos:
            break
        for rel, types, full in iter_instruction_files(repo):
            n_commits = commit_count_for_path(repo, rel)
            ts = last_commit_ts(repo, rel)
            try:
                size = full.stat().st_size
            except Exception:
                size = None
            for t in types:
                rows.append({
                    "repo": repo.name,
                    "cohort": cohort,
                    "file_type": t,
                    "path": rel,
                    "n_commits": n_commits,
                    "last_commit": ts.isoformat() if ts else None,
                    "bytes": size,
                })
    return pd.DataFrame(rows)


maint = maintenance_pilot("top", n_repos=40)
print(maint.groupby("file_type")["n_commits"].describe().round(2))
maint.sort_values("n_commits", ascending=False).head(15)

## Template synthesis map (filled as RQs complete)

| Evidence | Decision rule | Template slot |
|----------|---------------|---------------|
| RQ2 shared core (≥3 formats) **and** RQ3 low deletion churn | Keep | **REQUIRED** |
| RQ2 format-unique | Offer | Optional extension (Cursor / Claude / Copilot / AGENTS) |
| RQ2 gap (missing across types) | Add | Recommended addition |
| RQ3 ephemeral (high deletion / abandoned) | Demote | Optional / "only if needed" |
| RQ3 reactive trigger dominant | Add | Maintenance guidance section |
| RQ1 multi-file duplication | Consolidate | Single source of truth + tool-specific thin adapters |

## Data gaps / next actions

1. **Load AIDev-pop** (`hao-li/AIDev`) tables: `repository`, `pull_request`, `pr_commit_details`, `pr_reviews`, `pr_task_type`.
2. Filter `pr_commit_details.filename` to the four instruction path patterns → agentic-PR touch events for RQ1/RQ3.
3. Expand RQ1 drift + RQ3 maintenance from sample → full cloned corpus.
4. Run dual-coder card sort (RQ2) and trigger coding (RQ3); compute κ.
5. Emit the universal template draft from the synthesis map.

In [ ]:
# Optional: pull AIDev-pop from Hugging Face (large download)
# from datasets import load_dataset
# ds = load_dataset("hao-li/AIDev", "AIDev-pop")  # check config name on the hub
# pr_commit_details = ds["pr_commit_details"].to_pandas()
# instruction_touches = pr_commit_details[
#     pr_commit_details["filename"].fillna("").map(lambda p: bool(classify_path(p)))
# ]
print("Uncomment above once HF datasets + AIDev-pop are available on this machine.")